# 01 · Entrada de dados, ETL e monitor de drift

**Ciclo do pipeline MLOps (Trabalho N3).** Este notebook abre o ciclo: incorpora o próximo lote de acidentes,
gera o `cycle_id` e a `data_version`, diagnostica e trata os dados, monta a base trecho × dia × período e faz a
divisão temporal. No fim, roda o monitor de drift (critério 10).

**Problema (etapa 1):** classificação binária: haverá **ao menos um acidente** num trecho de 10 km de BR em SC,
num dia e num período de 6 h? Unidade de observação: trecho × dia × período. Fonte: PRF Dados Abertos
(acidentes 2017–2025, agrupados por pessoa), clima Open-Meteo (CC BY 4.0), VMD do DNIT e geometria do
OpenStreetMap (ODbL). Limitação: só acidentes **registrados** pela PRF.

**Avaliação definida antes do treino** (`config/config.yaml`): métrica principal **PR-AUC** (2% de positivos;
acurácia não serve), divisão **temporal**: validação = 6 meses finais dos dados do ciclo, treino = tudo antes;
2025 fica reservado para a avaliação final (notebook 05) e nunca entra numa decisão.

**Chegada de novos registros:** replay cronológico. A base inicial vai até dez/2023 e cada ciclo agendado
incorpora o próximo mês de 2024 (`data/raw/lotes/`), deduplicando pelo `id` do acidente.

In [1]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
from src import ingestao, preprocessamento as pp, drift
print(json.dumps({k: cfg[k] for k in ["avaliacao", "dados"]}, indent=2, ensure_ascii=False))

{
  "avaliacao": {
    "tipo": "classificacao",
    "alvo": "teve_acidente",
    "metrica": "pr_auc",
    "direcao": "maximize",
    "validacao_meses": 6
  },
  "dados": {
    "base_inicial": "data/raw/base_inicial.parquet",
    "lotes": "data/raw/lotes",
    "teste_reservado": "data/raw/teste_reservado.parquet",
    "trechos": "data/raw/trechos.parquet",
    "clima": "data/raw/clima.parquet",
    "inicio_lotes": "2024-01",
    "inicio_teste": "2025-01-01",
    "lotes_por_ciclo": 1
  }
}


## Entrada e versionamento do lote

In [3]:
ciclo, brutos = ingestao.iniciar_ciclo(cfg)
ciclo["gatilho"] = gatilho()
print(json.dumps(ciclo, indent=2, ensure_ascii=False))

{
  "cycle_id": "20261127-1856-46053022288",
  "data_version": "v2024-04-dc968cc9",
  "iniciado_em": "2026-11-27T18:56:49+00:00",
  "lotes": [
    "2024-01",
    "2024-02",
    "2024-03",
    "2024-04"
  ],
  "lotes_novos": [
    "2024-04"
  ],
  "fim_dos_dados": "2024-04-30",
  "linhas_brutas": 145571,
  "linhas_brutas_novas": 1599,
  "duplicatas_removidas": 139468,
  "primeiro_ciclo": false,
  "gatilho": "schedule"
}


## Diagnóstico e tratamento

Os lotes chegam **brutos**, como a PRF publica: uma linha por **pessoa** envolvida, todas as colunas como texto,
números com vírgula decimal. O tratamento é refeito a cada ciclo, em três passos:

1. **Diagnóstico bruto**: linhas repetidas, acidentes sem localização, coordenadas fora de SC, datas inválidas.
2. **Limpeza da PRF** (`backend/app/dados.limpar_pessoas`, a mesma do modelo da interface): converte números;
   soma as vítimas por acidente e mantém o tipo principal (`ordem_tipo_acidente = 1`), ficando com uma linha por
   acidente; descarta acidentes sem BR/km/coordenada ou fora de SC; padroniza datas (a PRF usou dois formatos);
   cria período do dia (madrugada/manhã/tarde/noite) e trecho de 10 km.
3. **Tratamento do ciclo** (`pp.tratar`): só trechos conhecidos e datas até o fim do ciclo; vítimas ausentes = 0.
   Acidentes com mais de 10 mortos são sinalizados como outlier, mas mantidos (são reais, e o alvo é binário).

In [4]:
trechos = pd.read_parquet(caminho(cfg["dados"]["trechos"]))
clima = pd.read_parquet(caminho(cfg["dados"]["clima"]))
fim = pd.Timestamp(ciclo["fim_dos_dados"])
from app import dados

diag_bruto = pp.diagnostico_bruto(brutos)
acidentes = dados.limpar_pessoas(brutos)
antes = pp.diagnostico(acidentes, trechos)
limpos = pp.tratar(acidentes, trechos, fim)
depois = pp.diagnostico(limpos, trechos)
print(f"linhas repetidas removidas na ingestão: {ciclo['duplicatas_removidas']}")
pd.Series(diag_bruto, name="dados brutos (uma linha por pessoa)").to_frame()

  10 acidentes com coordenada fora de SC descartados
linhas repetidas removidas na ingestão: 139468


,dados brutos (uma linha por pessoa)
linhas (pessoas),145571.00
acidentes distintos,60823.00
pessoas por acidente (média),2.39
linhas repetidas (id + pesid),0.00
acidentes sem br/km/coordenada,76.00
acidentes com coordenada fora de SC,10.00
datas em formato inválido,0.00
horário ausente,0.00
tipo de acidente ausente,8.00


In [5]:
pd.DataFrame({"após limpeza da PRF": pd.Series(antes, dtype=object),
              "após tratamento do ciclo": pd.Series(depois, dtype=object)})

,após limpeza da PRF,após tratamento do ciclo
linhas,60737,60737
ausentes,"{'tipo_acidente': 4, 'classificacao_acidente':...","{'tipo_acidente': 4, 'classificacao_acidente':..."
ids_duplicados,0,0
coordenada_fora_de_sc,0,0
km_negativo,0,0
trecho_desconhecido,0,0
periodo_invalido,0,0
vitimas_negativas,0,0
outlier_mortos_gt_10,0,0
periodo_dados,"[2017-01-01, 2024-04-30]","[2017-01-01, 2024-04-30]"


## Base trecho × dia × período e divisão temporal

`montar_base` (de `backend/scripts/treinar.py`, a mesma usada no modelo da interface) cria todas as combinações
trecho × dia × período — inclusive os trechos sem acidente —, junta clima e calendário e calcula o histórico do
trecho **só até o dia anterior** (sem vazamento). Imputação, codificação e escala ficam dentro dos `Pipeline`s
do notebook 03, ajustados só no treino.

In [6]:
base = pp.montar(limpos, trechos, clima)
treino, validacao, inicio_val = pp.dividir(base, fim, cfg["avaliacao"]["validacao_meses"])
resumo = pd.DataFrame({
    "linhas": [len(treino), len(validacao)],
    "inicio": [treino["data"].min().date(), validacao["data"].min().date()],
    "fim": [treino["data"].max().date(), validacao["data"].max().date()],
    "taxa_positivos": [round(treino[pp.ALVO].mean(), 4), round(validacao[pp.ALVO].mean(), 4)],
}, index=["treino", "validacao"])
resumo

,linhas,inicio,fim,taxa_positivos
treino,2305740,2017-06-30,2023-10-31,0.0209
validacao,181272,2023-11-01,2024-04-30,0.0212


In [7]:
pasta = estado(cfg, "processados"); pasta.mkdir(parents=True, exist_ok=True)
treino.to_parquet(pasta / "treino.parquet"); validacao.to_parquet(pasta / "validacao.parquet")
print("ausentes nas features:", base[pp.CANDIDATAS].isna().sum()[lambda s: s > 0].to_dict(),
      "(VMD: trechos sem segmento do DNIT; imputado no Pipeline)")

ausentes nas features: {'vmd': 139832} (VMD: trechos sem segmento do DNIT; imputado no Pipeline)


## Dicionário de atributos (candidatos à votação do notebook 02)

| Feature | Tipo | Descrição | Origem |
|---|---|---|---|
| `trecho_km` | numérica | km inicial do trecho de 10 km | PRF |
| `lat`, `lon` | numérica | ponto representativo do trecho | PRF/OSM |
| `br` | categórica | número da rodovia federal | PRF |
| `delegacia` | categórica | delegacia da PRF responsável pelo trecho | PRF |
| `vmd` | numérica | volume médio diário de veículos (ausente onde não há segmento do DNIT) | DNIT |
| `periodo_idx` | ordinal | 0 madrugada, 1 manhã, 2 tarde, 3 noite | calendário |
| `dia_semana`, `mes` | ordinal | dia da semana (0 = segunda) e mês | calendário |
| `fim_de_semana`, `feriado`, `vespera_ou_pos_feriado` | binária | feriados nacionais, incluindo os móveis | calendário |
| `temporada_verao` | binária | dez–fev (temporada no litoral) | calendário |
| `pandemia` | binária | 15/03/2020 a 31/12/2021 (menos tráfego) | calendário |
| `chuva_mm`, `nebulosidade`, `temperatura` | numérica | clima da célula de ~25 km no período (reanálise ERA5) | Open-Meteo |
| `taxa_historica` | numérica | acidentes por dia no trecho até a véspera | PRF (defasado) |
| `taxa_historica_periodo` | numérica | o mesmo, só no período do dia | PRF (defasado) |
| `acid_ult_365d` | numérica | acidentes no trecho nos 365 dias anteriores | PRF (defasado) |

Alvo: `teve_acidente` = 1 se houve ao menos um acidente no trecho, dia e período.

## Monitor de drift (critério 10)

Compara o mês mais recente do ciclo com o **mesmo mês dos 3 anos anteriores** (referência sazonal) pelo PSI de
cada feature e pela taxa de acidentes. Se passar dos limites do `config.yaml`, o workflow escreve no resumo da
execução e abre uma issue no GitHub.

In [8]:
mes = fim.to_period("M").to_timestamp()
referencia, atual = drift.recorte_mensal(base, mes)
rel_drift = drift.monitorar(referencia, atual, pp.CANDIDATAS, cfg)
print(drift.alertar(rel_drift, ciclo, str(mes.to_period("M"))))
pd.Series(rel_drift["psi"], name="PSI").to_frame().head(8)

## Monitor de drift · ciclo `20261127-1856-46053022288` · 2024-04
- data_version: `v2024-04-dc968cc9`
- PSI máximo: **0.468**; taxa de acidentes 1.014× a referência
- Alerta: **não**



,PSI
temperatura,0.4682
nebulosidade,0.1187
chuva_mm,0.0425
vespera_ou_pos_feriado,0.0392
acid_ult_365d,0.0266
feriado,0.0243
dia_semana,0.0231
taxa_historica_periodo,0.0154


## Registro no MLflow e estado do ciclo

In [9]:
ev = pasta_evidencias(cfg, ciclo)
(ev / "diagnostico.json").write_text(json.dumps({"bruto": diag_bruto, "apos_limpeza_prf": antes, "apos_tratamento": depois},
                                                indent=2, ensure_ascii=False, default=str))
(ev / "drift.json").write_text(json.dumps(rel_drift, indent=2, ensure_ascii=False))

mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · etl", tags=tags_ciclo(ciclo, etapa="etl")) as run:
    mlflow.log_params({"lotes_novos": ",".join(ciclo["lotes_novos"]) or "nenhum", "n_lotes": len(ciclo["lotes"]),
                       "fim_dos_dados": ciclo["fim_dos_dados"], "inicio_validacao": str(inicio_val.date())})
    mlflow.log_metrics({"linhas_brutas": ciclo["linhas_brutas"], "linhas_brutas_novas": ciclo["linhas_brutas_novas"],
                        "duplicatas_removidas": ciclo["duplicatas_removidas"], "acidentes_limpos": len(limpos),
                        "acidentes_descartados": diag_bruto["acidentes distintos"] - len(limpos),
                        "linhas_treino": len(treino), "linhas_validacao": len(validacao),
                        "taxa_treino": treino[pp.ALVO].mean(), "taxa_validacao": validacao[pp.ALVO].mean(),
                        "drift_psi_max": rel_drift["psi_max"], "drift_razao_taxa": rel_drift["razao_taxa"],
                        "alerta_drift": int(rel_drift["alerta"])})
    mlflow.log_artifact(ev / "diagnostico.json"); mlflow.log_artifact(ev / "drift.json")

ciclo = atualizar_ciclo(cfg, gatilho=ciclo["gatilho"], inicio_validacao=str(inicio_val.date()),
                        run_etl=run.info.run_id, alerta_drift=rel_drift["alerta"])
ciclo

{'cycle_id': '20261127-1856-46053022288',
 'data_version': 'v2024-04-dc968cc9',
 'iniciado_em': '2026-11-27T18:56:49+00:00',
 'lotes': ['2024-01', '2024-02', '2024-03', '2024-04'],
 'lotes_novos': ['2024-04'],
 'fim_dos_dados': '2024-04-30',
 'linhas_brutas': 145571,
 'linhas_brutas_novas': 1599,
 'duplicatas_removidas': 139468,
 'primeiro_ciclo': False,
 'gatilho': 'schedule',
 'inicio_validacao': '2023-11-01',
 'run_etl': 'c3debfa657304a4b991e233d7ed2e58f',
 'alerta_drift': False}